In [ ]:
# Base imports
import os
import pickle

# Compute imports
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm, trange

# Plotting imports
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from plotly import express as px

# ML import
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error, median_absolute_error
from sklearn.metrics.pairwise import cosine_similarity


import multiprocessing
from multiprocessing import Pool


matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
DF_GENES = '../../data/processed/panaroo_output/gene_presence_absence.Rtab'
ENRICHED_METADATA = '../../data/metadata/enriched_metadata.csv'
DF_EGGNOG = '../../data/processed/df_eggnog.csv'

DF_CORE_COMPLETE = '../../data/processed/CAR_genomes/df_core_panaroo.pickle'
DF_ACC_COMPLETE = '../../data/processed/CAR_genomes/df_acc_panaroo.pickle'
DF_RARE_COMPLETE = '../../data/processed/CAR_genomes/df_rare_panaroo.pickle'

In [ ]:
df_rare = pd.read_pickle(DF_RARE_COMPLETE)
df_acc = pd.read_pickle(DF_ACC_COMPLETE)
df_core = pd.read_pickle(DF_CORE_COMPLETE)

In [ ]:
A_BINARIZED = '../../data/processed/nmf-outputs/A_binarized.csv'
A_binarized = pd.read_csv(A_BINARIZED, index_col=0)

In [ ]:
metadata = pd.read_csv(ENRICHED_METADATA, index_col=0, dtype='object')

display( metadata.shape, metadata.head())

In [ ]:
# Load in (full) P matrix
df_genes = pd.read_csv(DF_GENES, sep='\t', index_col='Gene')

# Filter metadata for Complete sequences only
metadata_complete = metadata[metadata.genome_status == 'Complete'] # filter for only Complete sequences

# Filter P matrix for Complete sequences only
df_genes_complete = df_genes[metadata_complete.genome_id].copy()
df_genes_complete.fillna(0, inplace=True) # replace N/A with 0
df_genes_complete = df_genes_complete.astype('int8') # densify & typecast to int8 for space and compute reasons
inCompleteseqs = df_genes_complete.sum(axis=1) > 0 # filter for genes found in complete sequences
df_genes_complete = df_genes_complete[inCompleteseqs]

df_genes_complete.shape

In [ ]:
# Load in eggNOG annotations
df_eggnog = pd.read_csv(DF_EGGNOG, index_col=0)
df_eggnog.fillna('-', inplace=True)

display(
    df_eggnog.shape,
    df_eggnog.head()
)

In [ ]:
core_genes = df_core.index

In [ ]:
acc_genes = df_acc.index
rare_genes = df_rare.index

In [ ]:
phylon_order = [
    'mobile-1',
    'mobile-4',
    'mobile-2',
    'mobile-3',
    'mobile-10',
    'mobile-7',
    'mobile-6',
    'mobile-5',
    'mobile-8',
    'mobile-9',
    'roggenkampii',
    'asburiae-1',
    'asburiae-2',
    'cancerogenous',
    'kobei',
    'bugandensis',
    'mori',
    'ludwigii',
    'cloacae',
    'hormaechei-steigerwaltii-2',
    'hormaechei-steigerwaltii-4',
    'hormaechei-steigerwaltii-1',
    'hormaechei-steigerwaltii-3',
    'hormaechei-hoffmannii-1',
    'hormaechei-hoffmannii-2',
    'hormaechei-hoffmannii-3',
    'hormaechei-hormaechei',
    'hormaechei-oharae',
    'hormaechei-xiangfangensis-2',
    'hormaechei-xiangfangensis-1',
    'hormaechei-xiangfangensis-3',
]

characterized_order = [x for x in phylon_order if 'mobile' not in x]

df = A_binarized.loc[characterized_order]

# List to store labels and column names
label_col = []
name_col = []

# Iterate over columns
for col in df.columns:
    # Find index where value is 1
    index = df.index[df[col] == 1].tolist()
    if index:
        label_col.append(index[0])  # Append the first index where value is 1
    else:
        label_col.append('None')  # If no 1 is found, append None
    name_col.append(col)

# Create a new DataFrame
output_df = pd.DataFrame({'Column': name_col, 'Label': label_col}).set_index('Column')

custom_colors = [
    # non hormaechei species
    "Green",
    'Blue',
    'navy',
    'Magenta',
    'Purple',
    'Cyan',
    'Tan',
    'Lime',
    'Pink',
    # hormaechei colors
    'firebrick',
    'maroon',
    'darkred',
    'brown',
    'Goldenrod',
    'DarkGoldenrod',
    'Gold',
    'Yellow',
    'Red',
    'Orange',
    'darkorange',
    'orangered',
]


clr = dict(zip(characterized_order + ["None"], custom_colors))
output_df['color'] = output_df.Label.map(clr)

# Core Genome Anlignment

Alignment results from panaroo were built into a phylogenetic tree using Fasttree and the following command:
```
fasttree -nosupport -gtr -nt ../panaroo_output/core_gene_alignment_filtered.aln > fastree_tree
```


# Plot as a circular tree

In [ ]:
df_species = metadata_complete[metadata_complete.genome_status == 'Complete'].loc[:,["genome_id", "genome_name"]]
df_species["species"] = df_species["genome_name"].apply(lambda x: x.split()[0]+" " +x.split()[1])
df_species.set_index('genome_id', inplace=True)
custom_colors_species = {'Enterobacter hormaechei': 'FireBrick',
 'Enterobacter cloacae': 'Pink',
 'Enterobacter sp.': 'SlateGray',
 'Enterobacter roggenkampii': 'Green',
 'Enterobacter kobei': 'Purple',
 'Enterobacter cancerogenus': 'Magenta',
 'Enterobacter bugandensis': 'Cyan',
 'Enterobacter asburiae': 'Blue',
 'Enterobacter ludwigii': 'Lime',
 'Enterobacter mori': '#F5F5DC',
 'Enterobacter xiangfangensis': 'Red'}

df_species['color'] = df_species.species.map(custom_colors_species)

In [ ]:
def get_strains(phylon, A_binarized = A_binarized):
    phylon_membership = A_binarized.loc[phylon]
    return (phylon_membership[phylon_membership == 1]).index

In [ ]:
df = A_binarized.loc[characterized_order]

# List to store labels and column names
label_col = []
name_col = []

# Iterate over columns
for col in df.columns:
    # Find index where value is 1
    index = df.index[df[col] == 1].tolist()
    if index:
        label_col.append(index[0])  # Append the first index where value is 1
    else:
        label_col.append('None')  # If no 1 is found, append None
    name_col.append(col)

# Create a new DataFrame
output_df = pd.DataFrame({'Column': name_col, 'Label': label_col}).set_index('Column')

clr = dict(zip(characterized_order + ["None"], custom_colors))
custom_colors = clr
output_df['color'] = output_df.Label.map(clr)

df_species = output_df

In [ ]:
L_BINARIZED = '../../data/processed/nmf-outputs/L_binarized.csv'
A_BINARIZED = '../../data/processed/nmf-outputs/A_binarized.csv'

# Load in L_binarized matrix
L_binarized = pd.read_csv(L_BINARIZED, index_col=0)
A_binarized = pd.read_csv(A_BINARIZED, index_col=0)

In [ ]:
import matplotlib.pyplot as plt
from pycirclize import Circos
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import LabelEncoder

# Initialize the Circos plot
circos, tv = Circos.initialize_from_tree("../../data/processed/fasttree_outputs/fastree_tree", 
    leaf_label_size=0,
    start=10,
    end=350,
    r_lim=(20, 55),
    line_kws=dict(color="black", lw=2),
    align_line_kws=dict(ls="dashdot", lw=.3, alpha=.5),
    align_leaf_label=True,
    ladderize=True
)


# Plot heatmap with various style
sector = circos.sectors[0]
sector.rect(r_lim=(55, 65), ec="grey", lw=1)

# get labels for the phylons to map to the phylon colors
label_encoder = LabelEncoder()
test_df = df_species.copy()
test_df['num_labels'] = label_encoder.fit_transform(df_species.Label)

heatmap_track1 = sector.add_track((55, 60))
phylon_colors = LinearSegmentedColormap.from_list("phylon_colors", test_df.loc[tv.leaf_labels].sort_values('num_labels').fillna('white').color.unique())
heatmap_track1.heatmap(test_df.loc[tv.leaf_labels].num_labels.values, cmap=phylon_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Phylon", r=heatmap_track1.r_center, size=8, color="black")

# list of colors for the mash clusters
color_list = [
    # --- 4 shades of dark yellow / gold ---
    "#C5A300",  # Dark Gold
    "#D4AF37",  # Metallic Gold
    "#E1B000",  # Rich Yellow-Gold
    "#FFCC00",  # Bright Gold

    # --- 3 shades of red/orange ---
    "#FF6600",  # Orange
    "#FF4500",  # Red-Orange
    "#E25822",  # Burnt Orange

    # --- 1 yellow ---
    "#FFFF33",  # Bright Yellow

    # --- 1 red ---
    "#FF0000",  # True Red

    # --- 3 shades of dark red ---
    "#CC0000",  # Dark Red
    "#990000",  # Deep Red
    "#660000",  # Very Dark Red

    # --- 1 purple ---
    "#8000FF",  # Vivid Purple

    # --- 1 grey ---
    "#A0A0A0",  # Medium Grey

    # --- 3 shades of green ---
    "#66B266",  # Medium Green
    "#339933",  # Deep Green
    "#99CC99",  # Light Green

    # --- 2 more greys ---
    "#B0B0B0",  # Light Grey
    "#D9D9D9",  # Pale Grey

    # --- 1 tan ---
    "#D2B48C",  # Tan

    # --- 6 shades of blue ---
    "#0033CC",  # Deep Blue
    "#3366CC",  # Medium Blue
    "#6699FF",  # Sky Blue
    "#80BFFF",  # Soft Blue
    "#A3C2FF",  # Pale Blue
    "#CCDFFF",  # Very Light Blue

    # --- 1 grey ---
    "#808080",  # Mid Grey

    # --- 1 bright lavender (neon) ---
    "#C77DFF",  # Neon Lavender

    # --- 1 grey ---
    "#C0C0C0",  # Silver Grey

    # --- 2 shades of neon blue ---
    "#00BFFF",  # Neon Blue
    "#1E90FF",  # Bright Neon Blue

    # --- 1 neon green ---
    "#39FF14",  # Neon Green

    # --- 2 shades of pink ---
    "#FF66B2",  # Pink
    "#FF99CC"   # Light Pink
]


data = metadata_complete.set_index('genome_id')
heatmap_track2 = sector.add_track((60, 65))
mash_cluster_colors = LinearSegmentedColormap.from_list("mash_cluster_colors", color_list)
heatmap_track2.heatmap(data.loc[tv.leaf_labels].complete_mash_cluster.values.astype(float), cmap=mash_cluster_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Mash Cluster", r=heatmap_track2.r_center, size=8, color="black")

### add row for traits of interest ###

## mobile phylon tracks
phylon_data = (A_binarized.loc['mobile-1', tv.leaf_labels] > 0).astype(int)
heatmap_mobile1 = sector.add_track((66, 68))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#FF6961"])
heatmap_mobile1.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-1", r=heatmap_mobile1.r_center, size=8, color="black")

phylon_data = (A_binarized.loc['mobile-2', tv.leaf_labels] > 0).astype(int)
heatmap_mobile2 = sector.add_track((68, 70))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#AEC6CF"])
heatmap_mobile2.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-2", r=heatmap_mobile2.r_center, size=8, color="black")

phylon_data = (A_binarized.loc['mobile-3', tv.leaf_labels] > 0).astype(int)
heatmap_mobile3 = sector.add_track((70, 72))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#77DD77"])
heatmap_mobile3.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-3", r=heatmap_mobile3.r_center, size=8, color="black")

## mobile phylon tracks
phylon_data = (A_binarized.loc['mobile-4', tv.leaf_labels] > 0).astype(int)
heatmap_mobile4 = sector.add_track((72, 74))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#B39EB5"])
heatmap_mobile4.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-4", r=heatmap_mobile4.r_center, size=8, color="black")


# trait 1
# pqq genes associated with pyrroloquinoline quinone
# only found in hormaechei cluster
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2245851/ - promotes plant growth, rhizosphere associated
cond = df_eggnog.loc[df_genes.index].Preferred_name.apply(lambda x : 'pqq' in x)
cond2 = df_eggnog.loc[df_genes.index][cond].index
inds = df_acc.loc[df_eggnog.loc[[x for x in cond2 if x in df_acc.index]].index, tv.leaf_labels].index
genetic_trait_data = (df_acc.loc[inds, tv.leaf_labels].sum() > 2).astype(int)
heatmap_track3 = sector.add_track((75, 76))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#FF0000"])
heatmap_track3.heatmap(genetic_trait_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("pqq Genes", r=heatmap_track3.r_center, size=8, color="black")

# trait 2
# genes associated with Salmochelin production
cond = df_eggnog.loc[df_genes.index].Preferred_name.apply(lambda x : 'iro' in x)
cond2 = df_eggnog.loc[df_genes.index][cond].index
inds = df_genes_complete.loc[df_eggnog.loc[[x for x in cond2 if x in df_genes_complete.index]].index, tv.leaf_labels].index
genetic_trait_data = (df_genes_complete.loc[inds, tv.leaf_labels].sum() > 0).astype(int)
heatmap_track4 = sector.add_track((77, 78))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#000000"])
heatmap_track4.heatmap(genetic_trait_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Salmochelin VFs", r=heatmap_track4.r_center, size=8, color="black")

# trait 3, arn operon associated with colistin resistance
cond = df_eggnog.loc[df_genes.index].Preferred_name.apply(lambda x : 'arn' in x)
cond2 = df_eggnog.loc[df_genes.index][cond].index
inds = df_acc.loc[df_eggnog.loc[[x for x in cond2 if x in df_acc.index]].index, tv.leaf_labels].index
genetic_trait_data = (df_acc.loc[inds, tv.leaf_labels].sum() > 6).astype(int)
heatmap_track5 = sector.add_track((79, 80))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#3333FF"])
heatmap_track5.heatmap(genetic_trait_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Arn Operon", r=heatmap_track5.r_center, size=8, color="black")


### End of adding traits of interest ###
# Create the Circos figure
fig = circos.plotfig(figsize=(20,10))

# # Plot legend for phylons
# line_handles = []

# for label, name, color in zip(test_df.loc[tv.leaf_labels].num_labels.unique(),test_df.loc[tv.leaf_labels].Label.unique(), test_df.loc[tv.leaf_labels].color.fillna('white').unique()):
#     line_handles.append(Line2D([], [], color=color, label=name, lw=4))

# line_legend = circos.ax.legend(
#     handles=line_handles,
#     bbox_to_anchor=(0.80, 1.02),
#     # loc="upper right",
#     fontsize=8,
#     title="Phylons",
#     handlelength=2,
#     ncols=1
# )
# circos.ax.add_artist(line_legend)

# # Plot legend for mash cluster
# line_handles = []

# for name, color in zip(list(range(int(data.loc[tv.leaf_labels].complete_mash_cluster.astype(float).max()))), color_list):
#     line_handles.append(Line2D([], [], color=color, label=name, lw=4))

# line_legend = circos.ax.legend(
#     handles=line_handles,
#     bbox_to_anchor=(0, 1.02),
#     loc="upper left",
#     fontsize=8,
#     title="Mash Clusters",
#     handlelength=2,
#     ncols=1
# )
# circos.ax.add_artist(line_legend)

plt.title("Single-Copy Core Gene Phylogeny")
plt.savefig("circos_plot.png", format='png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pycirclize import Circos
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import LabelEncoder

# Initialize the Circos plot
circos, tv = Circos.initialize_from_tree("../../data/processed/fasttree_outputs/fastree_tree", 
    leaf_label_size=1,
    start=10,
    end=350,
    r_lim=(20, 55),
    line_kws=dict(color="black", lw=2),
    align_line_kws=dict(ls="dashdot", lw=.3, alpha=.5),
    align_leaf_label=True,
    ladderize=True
)


# Plot heatmap with various style
sector = circos.sectors[0]
sector.rect(r_lim=(55, 65), ec="grey", lw=1)

# get labels for the phylons to map to the phylon colors
label_encoder = LabelEncoder()
test_df = df_species.copy()
test_df['num_labels'] = label_encoder.fit_transform(df_species.Label)

heatmap_track1 = sector.add_track((55, 60))
phylon_colors = LinearSegmentedColormap.from_list("phylon_colors", test_df.loc[tv.leaf_labels].sort_values('num_labels').fillna('white').color.unique())
heatmap_track1.heatmap(test_df.loc[tv.leaf_labels].num_labels.values, cmap=phylon_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Phylon", r=heatmap_track1.r_center, size=8, color="black")

# list of colors for the mash clusters
color_list = [
    # --- 4 shades of dark yellow / gold ---
    "#C5A300",  # Dark Gold
    "#D4AF37",  # Metallic Gold
    "#E1B000",  # Rich Yellow-Gold
    "#FFCC00",  # Bright Gold

    # --- 3 shades of red/orange ---
    "#FF6600",  # Orange
    "#FF4500",  # Red-Orange
    "#E25822",  # Burnt Orange

    # --- 1 yellow ---
    "#FFFF33",  # Bright Yellow

    # --- 1 red ---
    "#FF0000",  # True Red

    # --- 3 shades of dark red ---
    "#CC0000",  # Dark Red
    "#990000",  # Deep Red
    "#660000",  # Very Dark Red

    # --- 1 purple ---
    "#8000FF",  # Vivid Purple

    # --- 1 grey ---
    "#A0A0A0",  # Medium Grey

    # --- 3 shades of green ---
    "#66B266",  # Medium Green
    "#339933",  # Deep Green
    "#99CC99",  # Light Green

    # --- 2 more greys ---
    "#B0B0B0",  # Light Grey
    "#D9D9D9",  # Pale Grey

    # --- 1 tan ---
    "#D2B48C",  # Tan

    # --- 6 shades of blue ---
    "#0033CC",  # Deep Blue
    "#3366CC",  # Medium Blue
    "#6699FF",  # Sky Blue
    "#80BFFF",  # Soft Blue
    "#A3C2FF",  # Pale Blue
    "#CCDFFF",  # Very Light Blue

    # --- 1 grey ---
    "#808080",  # Mid Grey

    # --- 1 bright lavender (neon) ---
    "#C77DFF",  # Neon Lavender

    # --- 1 grey ---
    "#C0C0C0",  # Silver Grey

    # --- 2 shades of neon blue ---
    "#00BFFF",  # Neon Blue
    "#1E90FF",  # Bright Neon Blue

    # --- 1 neon green ---
    "#39FF14",  # Neon Green

    # --- 2 shades of pink ---
    "#FF66B2",  # Pink
    "#FF99CC"   # Light Pink
]


data = metadata_complete.set_index('genome_id')
heatmap_track2 = sector.add_track((60, 65))
mash_cluster_colors = LinearSegmentedColormap.from_list("mash_cluster_colors", color_list)
heatmap_track2.heatmap(data.loc[tv.leaf_labels].complete_mash_cluster.values.astype(float), cmap=mash_cluster_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Mash Cluster", r=heatmap_track2.r_center, size=8, color="black")

### add row for traits of interest ###

## mobile phylon tracks
phylon_data = (A_binarized.loc['mobile-1', tv.leaf_labels] > 0).astype(int)
heatmap_mobile1 = sector.add_track((66, 68))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#FF6961"])
heatmap_mobile1.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-1", r=heatmap_mobile1.r_center, size=8, color="black")

phylon_data = (A_binarized.loc['mobile-2', tv.leaf_labels] > 0).astype(int)
heatmap_mobile2 = sector.add_track((68, 70))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#AEC6CF"])
heatmap_mobile2.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-2", r=heatmap_mobile2.r_center, size=8, color="black")

phylon_data = (A_binarized.loc['mobile-3', tv.leaf_labels] > 0).astype(int)
heatmap_mobile3 = sector.add_track((70, 72))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#77DD77"])
heatmap_mobile3.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-3", r=heatmap_mobile3.r_center, size=8, color="black")

## mobile phylon tracks
phylon_data = (A_binarized.loc['mobile-4', tv.leaf_labels] > 0).astype(int)
heatmap_mobile4 = sector.add_track((72, 74))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#B39EB5"])
heatmap_mobile4.heatmap(phylon_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("mobile-4", r=heatmap_mobile4.r_center, size=8, color="black")


# trait 1
# pqq genes associated with pyrroloquinoline quinone
# only found in hormaechei cluster
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2245851/ - promotes plant growth, rhizosphere associated
cond = df_eggnog.loc[df_genes.index].Preferred_name.apply(lambda x : 'pqq' in x)
cond2 = df_eggnog.loc[df_genes.index][cond].index
inds = df_acc.loc[df_eggnog.loc[[x for x in cond2 if x in df_acc.index]].index, tv.leaf_labels].index
genetic_trait_data = (df_acc.loc[inds, tv.leaf_labels].sum() > 2).astype(int)
heatmap_track3 = sector.add_track((75, 76))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#FF0000"])
heatmap_track3.heatmap(genetic_trait_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("pqq Genes", r=heatmap_track3.r_center, size=8, color="black")

# trait 2
# genes associated with Salmochelin production
cond = df_eggnog.loc[df_genes.index].Preferred_name.apply(lambda x : 'iro' in x)
cond2 = df_eggnog.loc[df_genes.index][cond].index
inds = df_genes_complete.loc[df_eggnog.loc[[x for x in cond2 if x in df_genes_complete.index]].index, tv.leaf_labels].index
genetic_trait_data = (df_genes_complete.loc[inds, tv.leaf_labels].sum() > 0).astype(int)
heatmap_track4 = sector.add_track((77, 78))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#000000"])
heatmap_track4.heatmap(genetic_trait_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Salmochelin VFs", r=heatmap_track4.r_center, size=8, color="black")

# trait 3, arn operon associated with colistin resistance
cond = df_eggnog.loc[df_genes.index].Preferred_name.apply(lambda x : 'arn' in x)
cond2 = df_eggnog.loc[df_genes.index][cond].index
inds = df_acc.loc[df_eggnog.loc[[x for x in cond2 if x in df_acc.index]].index, tv.leaf_labels].index
genetic_trait_data = (df_acc.loc[inds, tv.leaf_labels].sum() > 6).astype(int)
heatmap_track5 = sector.add_track((79, 80))
trait_colors = LinearSegmentedColormap.from_list("trait_colors", ["#FFFFFF", "#3333FF"])
heatmap_track5.heatmap(genetic_trait_data.loc[tv.leaf_labels].values, cmap=trait_colors, rect_kws=dict(ec="lightgrey", lw=0))
circos.text("Arn Operon", r=heatmap_track5.r_center, size=8, color="black")


### End of adding traits of interest ###
# Create the Circos figure
fig = circos.plotfig(figsize=(20,10))

# Plot legend for phylons
line_handles = []

for label, name, color in zip(test_df.loc[tv.leaf_labels].num_labels.unique(),test_df.loc[tv.leaf_labels].Label.unique(), test_df.loc[tv.leaf_labels].color.fillna('white').unique()):
    line_handles.append(Line2D([], [], color=color, label=name, lw=4))

line_legend = circos.ax.legend(
    handles=line_handles,
    bbox_to_anchor=(0.80, 1.02),
    # loc="upper right",
    fontsize=8,
    title="Phylons",
    handlelength=2,
    ncols=1
)
circos.ax.add_artist(line_legend)

# Plot legend for mash cluster
line_handles = []

for name, color in zip(list(range(int(data.loc[tv.leaf_labels].complete_mash_cluster.astype(float).max()))), color_list):
    line_handles.append(Line2D([], [], color=color, label=name, lw=4))

line_legend = circos.ax.legend(
    handles=line_handles,
    bbox_to_anchor=(0, 1.02),
    loc="upper left",
    fontsize=8,
    title="Mash Clusters",
    handlelength=2,
    ncols=1
)
circos.ax.add_artist(line_legend)

plt.title("Single-Copy Core Gene Phylogeny")
plt.savefig("circos_plot_labeled.svg", format='svg', dpi=600, bbox_inches='tight')
plt.show()